## 依赖导入

本例集合演示常微分/差分方程模型的数值求解与参数拟合：用 `scipy.integrate.odeint` 积分方程组（Logistic、Lorenz、三体轨道、追及与拦截问题等），用 `scipy.optimize.curve_fit`/`minimize` 拟合与优化参数，并用 `numpy`、`pandas`、`matplotlib` 处理数据与绘图。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import odeint
from scipy.optimize import curve_fit, minimize

## 全局绘图配置

用 `plt.rc` 设置全局绘图：启用 TeX 渲染数学公式（`usetex=True`），统一中文字体与字号（字号最终为 15），保证后续各图的公式标签正常显示。

In [ ]:
plt.rc("text", usetex=True)
plt.rc("font", size=18)
plt.rc("font", size=16)
plt.rc("font", size=15)

## 6.1  Logistic 微分方程解曲线

对 Logistic 增长 $i(t)=\dfrac{1}{1+(1/i_0-1)e^{-rt}}$（$i_0=0.1, r=0.01$），左子图画其导数 $\dfrac{di}{dt}=r\,i(1-i)$、右子图画 $i(t)$ 随 $t\in[0,800]$ 的曲线，二者互相印证（源程序 zex8_1.py）。

In [ ]:
i0 = 0.1
r = 0.01


def it(t):
    return 1 / (1 + (1 / i0 - 1) * np.exp(-r * t))


def dit(t):
    return r * it(t) * (1 - it(t))


t = np.linspace(0, 800, 801)
fig, (ax, ax2) = plt.subplots(1, 2)
ax.plot(t, dit(t))
ax.set(xlabel="$t$")
ax.set_ylabel("$\\frac{di}{dt}$", rotation=0)
ax2.plot(t, it(t))
ax2.set(xlabel="$t$")
ax2.set_ylabel("$i(t)$", rotation=0)
plt.show()

## 6.8  悬链线型方程数值积分

把二阶方程化为一阶方程组 `yx`（初值 $[0,0]$），用 `odeint` 在 $x\in[0,1)$、步长 $10^{-5}$ 上积分，绘制 $y_0(x)$ 曲线（源程序 ex8_8.py）。

In [ ]:
def yx(y, x):
    return [y[1], np.sqrt(1 + y[1] ** 2) / 5 / (1 - x)]


x0 = np.arange(0, 1, 0.00001)
y0 = odeint(yx, [0, 0], x0)
fig, ax = plt.subplots()
ax.plot(x0, y0[:, 0])
plt.show()

## 6.9  Lorenz 吸引子与初值敏感性

用 `odeint` 积分 Lorenz 方程组 `g`（$\sigma=10,\rho=28,\beta=8/3$）。先 `np.random.seed(2)` 固定随机初值 $s_{01}$ 以保证每次运行可比；左子图用 3D 绘制轨线 $(x,y,z)$。再取扰动初值 $s_{02}=s_{01}+10^{-6}$ 积分，右子图绘制两条轨线 $x_1(t)-x_2(t)$ 之差随 $t$ 的增长，展示混沌系统的初值敏感性（源程序 ex8_9.py）。

In [ ]:
np.random.seed(2)  # noqa: NPY002
sigma = 10
rho = 28
beta = 8 / 3


def g(f, t):
    return [
        sigma * (f[1] - f[0]),
        rho * f[0] - f[1] - f[0] * f[2],
        f[0] * f[1] - beta * f[2],
    ]


s01 = np.random.rand(3)  # noqa: NPY002
t0 = np.linspace(0, 50, 5000)
s1 = odeint(g, s01, t0)
fig = plt.figure()
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot(s1[:, 0], s1[:, 1], s1[:, 2], "r")
ax.set(xlabel="$x$", ylabel="$y$", zlabel="$z$")
s02 = s01 + 0.000001
s2 = odeint(g, s02, t0)
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(t0, s1[:, 0] - s2[:, 0], ".-")
ax2.set(xlabel="$t$")
ax2.set_ylabel("$x_1(t)-x_2(t)$", rotation=90)
plt.show()

## 6.10  追击运动微分方程

`df` 定义追者沿视线方向以速度 $w$ 运动的方程组，`d` 为两者距离函数。分别在 $t_1\in[0,3.4068]$（$w=20$，终值时间逐步凑出）与 $t_2\in[0,50]$（$w=5$）上积分，绘制两张距离 $d$ 随时间变化的曲线，比较可截获与追不上的情形（源程序 ex8_10.py）。

In [ ]:
def df(f, t, w):
    return [
        w
        / np.sqrt((10 + 20 * np.cos(t) - f[0]) ** 2 + (20 + 15 * np.sin(t) - f[1]) ** 2)
        * (10 + 20 * np.cos(t) - f[0]),
        w
        / np.sqrt((10 + 20 * np.cos(t) - f[0]) ** 2 + (20 + 15 * np.sin(t) - f[1]) ** 2)
        * (20 + 15 * np.sin(t) - f[1]),
    ]


def d(xy, t):
    return np.sqrt(
        (xy[:, 0] - 10 - 20 * np.cos(t)) ** 2 + (xy[:, 1] - 20 - 15 * np.sin(t)) ** 2
    )


t1 = np.linspace(0, 3.4068, 100)
s1 = odeint(df, [0, 0], t1, args=(20,))
d1 = d(s1, t1)
fig, ax = plt.subplots()
ax.plot(t1, d1)
ax.set(xlabel="$t$")
ax.set_ylabel("$d$", rotation=0)
fig2, ax2 = plt.subplots()
t2 = np.linspace(0, 50, 501)
s2 = odeint(df, [0, 0], t2, args=(5,))
d2 = d(s2, t2)
ax2.plot(t2, d2)
ax2.set(xlabel="$t$")
ax2.set_ylabel("$d$", rotation=0)
plt.show()

## 6.12  限制性三体问题的周期轨道

以地月质量比 $\mu=1/82.45$ 构造旋转坐标系下的 Hill 方程组 `dz`（4 维状态），用 `odeint` 从初值 $[1.2,0,0,-1.0494]$ 积分 $t\in[0,100]$，在 $(x,z)$ 平面绘制轨道曲线（源程序 ex8_12.py）。

In [ ]:
mu = 1 / 82.45
lamda = 1 - mu


def dz(z, t):
    return [
        z[1],
        2 * z[3]
        + z[0]
        - lamda * (z[0] + mu) / ((z[0] + mu) ** 2 + z[2] ** 2) ** (3 / 2)
        - mu * (z[0] - lamda) / ((z[0] + lamda) ** 2 + z[2] ** 2) ** (3 / 2),
        z[3],
        -2 * z[1]
        + z[2]
        - lamda * z[2] / ((z[0] + mu) ** 2 + z[2] ** 2) ** (3 / 2)
        - mu * z[2] / ((z[0] + lamda) ** 2 + z[2] ** 2) ** (3 / 2),
    ]


t = np.linspace(0, 100, 1001)
s = odeint(dz, [1.2, 0, 0, -1.0494], t)
fig, ax = plt.subplots()
ax.plot(s[:, 0], s[:, 2])
plt.show()

## 6.13  美国人口 Logistic 拟合

读取 `data/8_13.xlsx`，隔行提取有效人口数据 `xd`、年份 `td`。方法一用 `curve_fit` 在参数边界 `bd` 下拟合 Logistic $x(t)$（初值 $x(1790)=3.9$）得 `p1` 并预测 2010 年值；方法二/三把模型线性化，分别以 $x_k$、$x_{k+1}$ 构造系数矩阵与常数项列（`b1`、`b2`），用 `np.linalg.pinv` 最小二乘估计 $r, x_m$，再预测 2010 年人口并横向比较（源程序 ex8_13.py）。

In [ ]:
a = pd.read_excel("data/8_13.xlsx", header=None)
b = a.values
xd = b[1::2, :]
xd = xd[~np.isnan(xd)]
td = np.linspace(1790, 2000, 22)
def x(t, r, xm):
    return xm / (1 + (xm / 3.9 - 1) * np.exp(-r * (t - 1790)))
bd = [(0, 200), (0.1, 1000)]
p1 = curve_fit(x, td[1:], xd[1:], bounds=bd)[0]
print(p1)
print("2010 年的预测值为：", round(x(2010, *p1), 4))
n = len(xd)
b1 = np.diff(xd) / 10 / xd[:-1]
a1 = np.vstack([np.ones(n - 1), -xd[:-1]]).T
p2 = np.linalg.pinv(a1) @ b1
r = p2[0]
xm = r / p2[1]
xh2 = x(2010, r, xm)
print("----------")
print(round(r, 4))
print(round(xm, 4))
print(round(xh2, 4))
b2 = np.diff(xd) / 10 / xd[1:]
a2 = np.vstack([np.ones(n - 1), -xd[1:]]).T
p3 = np.linalg.pinv(a2) @ b2
r = p3[0]
xm = r / p3[1]
xh3 = x(2010, r, xm)
print("----------")
print(round(r, 4))
print(round(xm, 4))
print(round(xh3, 4))

## 6.14  捕食者-食饵方程参数辨识

给定时刻 `t0` 与食饵 `x0`、捕食者 `y0` 观测，构造差分 $\Delta t, \Delta x, \Delta y$，把 Lotka-Volterra 方程离散为线性方程组：`mat1, mat2` 拼成系数矩阵 `mat`、`b` 为常数项列，用 `np.linalg.pinv` 最小二乘拟合模型参数 $a, b, c, d$ 并输出（源程序 ex8_14.py）。

In [ ]:
t0 = np.array([0, 1, 2, 3, 4, 5, 6, 8, 10, 12, 14, 16, 18])
x0 = np.array([60, 63, 64, 63, 61, 58, 53, 44, 39, 38, 41, 46, 53])
y0 = np.array([30, 34, 38, 44, 50, 55, 58, 56, 47, 38, 30, 27, 26])
dt = np.diff(t0)
dx = np.diff(x0)
dy = np.diff(y0)
temp = x0[:-1] * y0[:-1]
mat1 = np.vstack([x0[:-1], -temp, np.zeros((2, 12))]).T
mat2 = np.vstack([np.zeros((2, 12)), -y0[:-1], temp]).T
mat = np.vstack([mat1, mat2])
b = np.hstack([dx / dt, dy / dt])
cs = np.linalg.pinv(mat) @ b
print("参数 a,b,c,d 的值分别为:", np.round(cs, 4))

## 6.16  等额本息月供（差分方程解析解）

贷款 $Q=300000$、月利率 $r=0.051/12$、期数 $N=360$，用等额本息公式 $x=\dfrac{(1+r)^N Q r}{(1+r)^N-1}$ 计算月供（保留 2 位）并输出总还款额 $x\cdot N$（源程序 ex8_16.py）。

In [ ]:
Q = 300000
r = 0.051 / 12
N = 360
x = round((1 + r) ** N * Q * r / ((1 + r) ** N - 1), 2)
print(x)
print(x * N)

## 6.17  人口 Logistic 离散模型拟合

读取 `data/8_13.xlsx` 提取有效人口 `xd`，以 $x_{k+1}=a x_k + b x_k^2$ 建立线性模型（`A` 为系数矩阵），用 `np.linalg.pinv` 最小二乘拟合参数 `cs`，据此递推预测 2010 年人口 `xh` 并输出（源程序 ex8_17.py）。

In [ ]:
a = pd.read_excel("data/8_13.xlsx", header=None)
b = a.values
xd = b[1::2, :]
xd = xd[~np.isnan(xd)]
A = np.vstack([xd[:-1], xd[:-1] ** 2]).T
cs = np.linalg.pinv(A) @ xd[1:]
xh = cs[0] * xd[-1] + cs[1] * xd[-1] ** 2
print("参数值为：", np.round(cs, 4))
print("2010 年人口预测值为：", round(xh, 4))

## 6.18  追及问题的欧拉法数值解

追者速度 $u=5v$（$v=1$）、步长 $dt=4\times10^{-5}$，从原点 $(0,0)$ 出发，每步沿指向目标的视线方向用欧拉法更新 $(x,y)$ 与时间 $t$，同时逐点绘制轨线；当 $x$ 接近 1 时停止，输出 $x, y, t$（源程序 ex8_18.py）。

In [ ]:
v = 1
u = 5 * v
dt = 0.00004
x = 0
y = 0
t = 0
fig, ax = plt.subplots()
ax.plot(x, y, ".")
while x <= 0.99999:
    pq = [1 - x, v * t - y]
    x = x + u * pq[0] / np.linalg.norm(pq) * dt
    y = y + u * pq[1] / np.linalg.norm(pq) * dt
    t = t + dt
    ax.plot(x, y, ".")
print("x=", round(x, 4))
print("y=", round(y, 4))
print("t=", round(t, 4))
ax.set(xlabel="$x$")
ax.set_ylabel("$y$", rotation=0)
plt.show()

## 6.19  捕捞策略优化（两种建模）

渔场捕捞模型上半：枚举年捕捞死亡率参数 $k\in[0,0.875)$（步长 $10^{-3}$），由存活率 $a=1-0.2^{1/12}$ 递推四个年龄组鱼群量 $x_1\dots x_4$，累加得年产量序列 $Z$，用 `max`/`argmax` 找最大产量及对应的 $k_4=0.001\cdot\text{ind}$、$k_3=0.42k_4$，并绘制产量-捕捞强度曲线 $z(k)$（源程序 ex8_19_1.py）。

In [ ]:
a = 1 - 0.2 ** (1 / 12)
m = 1.109 * 10**5
w3 = 17.86
w4 = 22.99
X = []
Z = []
N = []
for k in np.arange(0, 0.875, 0.001):
    x1 = (
        1.22
        * 10**11
        * (
            1
            - 1
            / (
                m
                * (1 - a - 0.42 * k) ** 8
                * (1 - a) ** 24
                * (1 / 2 + (1 - a - k) ** 8 * (1 - a) ** 4)
            )
        )
    )
    x2 = (1 - a) ** 12 * x1
    x3 = (1 - a) ** 12 * x2
    x4 = (1 - a - 0.42 * k) ** 8 * (1 - a) ** 4 * x3
    X.append([x1, x2, x3, x4])
    n = (
        m
        * (1 - a - 0.42 * k) ** 8
        * (1 - a) ** 24
        * (1 / 2 + (1 - a - k) ** 8 * (1 - a) ** 4)
        * x1
    )
    N.append(n)
    z = (
        0.42 * k * w3 * (1 - (1 - a - 0.42 * k) ** 8) / (a + 0.42 * k) * x3
        + k * w4 * (1 - (1 - a - k) ** 8) / (a + k) * x4
    )
    Z.append(z)
mz = max(np.array(Z))
ind = np.argmax(Z)
k4 = 0.001 * ind
k3 = 0.42 * k4
print("最大生产量：", mz)
print("各年龄组鱼群数：", X[ind])
print("k4=", k4)
print("k3=", k3)
fig, ax = plt.subplots()
ax.plot(np.arange(0, 0.875, 0.001), Z)
ax.set_ylabel("$z$", rotation=0)
ax.set(xlabel="$k$")
plt.show()

下半改用 `minimize` 直接优化：决策向量 $z=[x_1,x_2,x_3,x_4,k]$，`n(z)` 为补充量、`b(z)` 为存活率，目标 `obj(z)` 取负年产量（等价于最大化产量）；四个等式约束 `con` 表达相邻年龄组的递推关系，`bd` 给定 $x_i$ 下界 $10^9$ 与 $k\in(0,1-a)$，随机初值 `np.random.rand(5)` 出发，打印最优解与最优值（源程序 ex8_19_2.py）。

In [ ]:
a = 1 - 0.2 ** (1 / 12)
m = 1.109 * 10**5
w = [5.07, 11.55, 17.86, 22.99]
def n(z):
    return m / 2 * (1 - a - 0.42 * z[4]) ** 8 * z[2] + m * (1 - a - z[4]) ** 8 * z[3]
def b(z):
    return 1.22 * 10**11 / (1.22 * 10**11 + n(z))
def obj(z):
    return -(
        0.42 * z[4] * w[2] * (1 - (1 - a - 0.42 * z[4]) ** 8) / (a + 0.42 * z[4]) * z[2]
        + z[4] * w[3] * (1 - (1 - a - z[4]) ** 8) / (a + z[4]) * z[3]
    )
con = [
    {"type": "eq", "fun": lambda z: z[0] - b(z) * n(z)},
    {"type": "eq", "fun": lambda z: z[1] - (1 - a) ** 12 * z[0]},
    {"type": "eq", "fun": lambda z: z[2] - (1 - a) ** 12 * z[1]},
    {
        "type": "eq",
        "fun": lambda z: z[3] - (1 - a - 0.42 * z[4]) ** 8 * (1 - a) ** 4 * z[2],
    },
]
bd = [(10**9, np.inf) for i in range(4)]
bd.append((0, 1 - a))
res = minimize(obj, 1000 * np.random.rand(5), constraints=con, bounds=bd)  # noqa: NPY002
print(res)
print("最优解：", np.round(res.x, 4))
print("目标函数的最优值：", round(res.fun, 4))